In [22]:
import pandas as pd
import numpy as np
import os
import shutil
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from scipy.stats import ttest_rel
from scipy.ndimage import zoom
from huggingface_hub import hf_hub_download

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [23]:
os.makedirs("oasis_images", exist_ok=True)

csv_cache_path = hf_hub_download(
    repo_id="pzarzycki/mri-oasis-1-ixi-pre",
    repo_type="dataset",
    filename="oasis_preprocessed/oasis_cross-sectional.csv"
)
local_csv_path = "oasis_cross-sectional.csv"
shutil.copyfile(csv_cache_path, local_csv_path)
print(f"CSV copied to: {local_csv_path}")

df_ids = pd.read_csv(local_csv_path)
subjects = df_ids['ID'].tolist()
print(f"Found {len(subjects)} subjects to download.")

# --- FIX: Use '_t1_masked.npy' (not '_tl_masked.npy') ---
for sub in subjects:
    try:
        img_path = hf_hub_download(
            repo_id="pzarzycki/mri-oasis-1-ixi-pre",
            repo_type="dataset",
            filename=f"oasis_preprocessed/{sub}_t1_masked.npy",   # <-- CORRECTED
            local_dir="./oasis_images",
            local_dir_use_symlinks=False
        )
        print(f"Downloaded: {sub}")
    except Exception as e:
        print(f"Error with {sub}: {e}")

CSV copied to: oasis_cross-sectional.csv
Found 436 subjects to download.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Downloaded: OAS1_0001_MR1
Downloaded: OAS1_0002_MR1
Downloaded: OAS1_0003_MR1
Downloaded: OAS1_0004_MR1
Downloaded: OAS1_0005_MR1
Downloaded: OAS1_0006_MR1
Downloaded: OAS1_0007_MR1
Downloaded: OAS1_0009_MR1
Downloaded: OAS1_0010_MR1
Downloaded: OAS1_0011_MR1
Downloaded: OAS1_0012_MR1
Downloaded: OAS1_0013_MR1
Downloaded: OAS1_0014_MR1
Downloaded: OAS1_0015_MR1
Downloaded: OAS1_0016_MR1
Downloaded: OAS1_0017_MR1
Downloaded: OAS1_0018_MR1
Downloaded: OAS1_0019_MR1
Downloaded: OAS1_0020_MR1
Downloaded: OAS1_0021_MR1
Downloaded: OAS1_0022_MR1
Downloaded: OAS1_0023_MR1
Downloaded: OAS1_0025_MR1
Downloaded: OAS1_0026_MR1
Downloaded: OAS1_0027_MR1
Downloaded: OAS1_0028_MR1
Downloaded: OAS1_0029_MR1
Downloaded: OAS1_0030_MR1
Downloaded: OAS1_0031_MR1
Downloaded: OAS1_0032_MR1
Downloaded: OAS1_0033_MR1
Downloaded: OAS1_0034_MR1
Downloaded: OAS1_0035_MR1
Downloaded: OAS1_0037_MR1
Downloaded: OAS1_0038_MR1
Downloaded: OAS1_0039_MR1
Downloaded: OAS1_0040_MR1
Downloaded: OAS1_0041_MR1
Downloaded: 

In [24]:
df = pd.read_csv("oasis_cross-sectional.csv")
df['target'] = (df['CDR'] > 0).astype(int)
df['Sex_encoded'] = df['M/F'].map({'M': 0, 'F': 1})

# Clean clinical columns (No MMSE)
clinical_columns = ['Age', 'Educ', 'Sex_encoded']

df['mri_path'] = df['ID'].apply(lambda x: os.path.join("oasis_images/oasis_preprocessed", f"{x}_t1_masked.npy"))
df['file_exists'] = df['mri_path'].apply(os.path.exists)
df = df[df['file_exists']].reset_index(drop=True)
print(f"Dataset size: {len(df)}")

Dataset size: 436


In [25]:
train_val_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target'])
train_df, val_df = train_test_split(train_val_df, test_size=0.2, random_state=42, stratify=train_val_df['target'])

# FIXED: Imputation strictly isolated to prevent leakage
imputer = SimpleImputer(strategy='median')
train_df[clinical_columns] = imputer.fit_transform(train_df[clinical_columns])
val_df[clinical_columns] = imputer.transform(val_df[clinical_columns])
test_df[clinical_columns] = imputer.transform(test_df[clinical_columns])

scaler = StandardScaler()
train_df[clinical_columns] = scaler.fit_transform(train_df[clinical_columns])
val_df[clinical_columns] = scaler.transform(val_df[clinical_columns])
test_df[clinical_columns] = scaler.transform(test_df[clinical_columns])

In [26]:
class OASISMultimodalDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        mri_volume = np.load(row['mri_path']).astype(np.float32)
        target_shape = (96, 128, 96)
        if mri_volume.shape != target_shape:
            mri_volume = zoom(mri_volume, [t/s for t, s in zip(target_shape, mri_volume.shape)], order=1)

        if mri_volume.max() > mri_volume.min():
            mri_volume = (mri_volume - mri_volume.min()) / (mri_volume.max() - mri_volume.min())
        mri_volume = np.expand_dims(mri_volume, axis=0)

        clinical_data = row[clinical_columns].values.astype(np.float32)
        label = np.array(row['target'], dtype=np.int64)

        return torch.tensor(mri_volume), torch.tensor(clinical_data), torch.tensor(label)

In [27]:
BATCH_SIZE = 4
train_loader = DataLoader(OASISMultimodalDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(OASISMultimodalDataset(val_df), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(OASISMultimodalDataset(test_df), batch_size=BATCH_SIZE, shuffle=False)

In [28]:
class MRI_CNN(nn.Module):
    def __init__(self, input_channels=1, feature_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv3d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool3d(1)
        )
        self.fc = nn.Linear(128, feature_dim)

    def forward(self, x):
        return self.fc(self.features(x).view(x.size(0), -1))

class Clinical_MLP(nn.Module):
    def __init__(self, input_dim=len(clinical_columns), hidden_dim=32, output_dim=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, output_dim), nn.ReLU())
    def forward(self, x): return self.net(x)

class MultimodalFusion(nn.Module):
    def __init__(self, mri_dim=128, clin_dim=32, num_classes=2):
        super().__init__()
        self.mri_enc = MRI_CNN(feature_dim=mri_dim)
        self.clin_enc = Clinical_MLP(output_dim=clin_dim)
        self.clf = nn.Sequential(nn.Linear(mri_dim + clin_dim, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, num_classes))
    def forward(self, mri, clin):
        return self.clf(torch.cat([self.mri_enc(mri), self.clin_enc(clin)], dim=1))

In [29]:
def process_epoch(model, loader, optimizer, criterion, mri_prob=1.0, clin_prob=1.0, clin_mean=None, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_probs, all_labels = 0, [], [], []

    with torch.set_grad_enabled(train):
        for mri, clinical, labels in loader:
            mri, clinical, labels = mri.to(device), clinical.to(device), labels.to(device)
            if mri_prob < 1.0: mri = mri * torch.bernoulli(torch.full((mri.size(0),1,1,1,1), mri_prob)).to(device)
            if clin_prob < 1.0 and clin_mean is not None:
                mask = torch.bernoulli(torch.full((mri.size(0),1), clin_prob)).to(device)
                clinical = torch.where(mask==1, clinical, clin_mean.expand_as(clinical).to(device))

            logits = model(mri, clinical)
            loss = criterion(logits, labels)
            total_loss += loss.item()

            if train: optimizer.zero_grad(); loss.backward(); optimizer.step()

            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            if not train: all_probs.extend(F.softmax(logits, dim=1)[:, 1].cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    if train: return total_loss/len(loader), acc
    return total_loss/len(loader), acc, f1_score(all_labels, all_preds, average='weighted', zero_division=0), roc_auc_score(all_labels, all_probs)

In [30]:
def run_experiment(exp_name, mri_present=True, clinical_present=True, epochs=30):
    print(f"\n{'='*40}\n{exp_name}")
    weights = compute_class_weight('balanced', classes=np.unique(train_df['target']), y=train_df['target'])
    crit = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float).to(device))
    c_mean = torch.tensor(train_df[clinical_columns].mean().values, dtype=torch.float32).to(device)

    model = MultimodalFusion().to(device)
    opt = optim.Adam(model.parameters(), lr=0.001)

    best_auc, pat = 0.0, 0
    for epoch in range(epochs):
        process_epoch(model, train_loader, opt, crit, mri_present, clinical_present, c_mean, True)
        _, _, _, v_auc = process_epoch(model, val_loader, opt, crit, mri_present, clinical_present, c_mean, False)
        if v_auc > best_auc: best_auc, pat = v_auc, 0; torch.save(model.state_dict(), f"{exp_name}.pth")
        else:
            pat += 1
            if pat >= 5: break

    model.load_state_dict(torch.load(f"{exp_name}.pth"))
    _, t_acc, t_f1, t_auc = process_epoch(model, test_loader, opt, crit, mri_present, clinical_present, c_mean, False)
    print(f"Test Acc: {t_acc:.4f} | F1: {t_f1:.4f} | AUC: {t_auc:.4f}")

In [32]:
run_experiment("Exp1_MRI", mri_present=1.0, clinical_present=0.0)
run_experiment("Exp2_Clinical", mri_present=0.0, clinical_present=1.0)
run_experiment("Exp3_Multi", mri_present=1.0, clinical_present=1.0)


Exp1_MRI
Test Acc: 0.7727 | F1: 0.6737 | AUC: 0.5676

Exp2_Clinical
Test Acc: 0.7273 | F1: 0.7492 | AUC: 0.8301

Exp3_Multi
Test Acc: 0.7500 | F1: 0.7681 | AUC: 0.8375


In [34]:
df.groupby('target')['Age'].describe()

,count,mean,std,min,25%,50%,75%,max
target,,,,,,,,
0,336.0,43.797619,23.754103,18.0,22.0,34.0,65.0,94.0
1,100.0,76.760000,7.119641,62.0,72.0,77.0,81.0,96.0


In [35]:
print("\n" + "="*60 + "\nLEAK-FREE K‑FOLD CV & T-TEST\n" + "="*60)

c_accs, c_f1s, c_aucs = [], [], []
m_accs, m_f1s, m_aucs = [], [], []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['target']), 1):
    print(f"\n--- Fold {fold} ---")
    train_f, val_f = df.iloc[train_idx].copy(), df.iloc[val_idx].copy()

    # Isolated Imputation & Scaling per fold
    imp, scl = SimpleImputer(strategy='median'), StandardScaler()
    train_f[clinical_columns] = scl.fit_transform(imp.fit_transform(train_f[clinical_columns]))
    val_f[clinical_columns] = scl.transform(imp.transform(val_f[clinical_columns]))

    tl = DataLoader(OASISMultimodalDataset(train_f), batch_size=4, shuffle=True)
    vl = DataLoader(OASISMultimodalDataset(val_f), batch_size=4, shuffle=False)

    c_mean = torch.tensor(train_f[clinical_columns].mean().values, dtype=torch.float32).to(device)
    w = compute_class_weight('balanced', classes=np.unique(train_f['target']), y=train_f['target'])
    crit = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float).to(device))

    # ---- Clinical Model ----
    model_c = MultimodalFusion().to(device)
    opt_c = optim.Adam(model_c.parameters(), lr=0.001)

    print("  Training Clinical...")
    # FIXED: Fixed 20 epochs, no early stopping double-dip
    for _ in range(20):
        process_epoch(model_c, tl, opt_c, crit, 0.0, 1.0, c_mean, True)

    # Blind evaluation on the held-out validation fold
    _, a, f, u = process_epoch(model_c, vl, opt_c, crit, 0.0, 1.0, c_mean, False)
    c_accs.append(a); c_f1s.append(f); c_aucs.append(u)
    print(f"  Clinical   -> AUC: {u:.4f}")

    # ---- Multimodal Model ----
    model_m = MultimodalFusion().to(device)
    opt_m = optim.Adam(model_m.parameters(), lr=0.001)

    print("  Training Multimodal...")
    # FIXED: Fixed 20 epochs, no early stopping double-dip
    for _ in range(20):
        process_epoch(model_m, tl, opt_m, crit, 1.0, 1.0, c_mean, True)

    # Blind evaluation on the held-out validation fold
    _, a, f, u = process_epoch(model_m, vl, opt_m, crit, 1.0, 1.0, c_mean, False)
    m_accs.append(a); m_f1s.append(f); m_aucs.append(u)
    print(f"  Multimodal -> AUC: {u:.4f}")

print("\n" + "="*60)
print(f"Clin  - Mean AUC: {np.mean(c_aucs):.4f}")
print(f"Multi - Mean AUC: {np.mean(m_aucs):.4f}")
t, p = ttest_rel(c_aucs, m_aucs)
print(f"Paired T-Test: t = {t:.4f}, p = {p:.4f}")
print("="*60)


LEAK-FREE K‑FOLD CV & T-TEST

--- Fold 1 ---
  Training Clinical...
  Clinical   -> AUC: 0.8809
  Training Multimodal...
  Multimodal -> AUC: 0.8801

--- Fold 2 ---
  Training Clinical...
  Clinical   -> AUC: 0.8836
  Training Multimodal...
  Multimodal -> AUC: 0.8910

--- Fold 3 ---
  Training Clinical...
  Clinical   -> AUC: 0.8519
  Training Multimodal...
  Multimodal -> AUC: 0.8504

--- Fold 4 ---
  Training Clinical...
  Clinical   -> AUC: 0.8944
  Training Multimodal...
  Multimodal -> AUC: 0.8959

--- Fold 5 ---
  Training Clinical...
  Clinical   -> AUC: 0.9243
  Training Multimodal...
  Multimodal -> AUC: 0.9276

Clin  - Mean AUC: 0.8870
Multi - Mean AUC: 0.8890
Paired T-Test: t = -1.2555, p = 0.2776
